In [46]:
!nvidia-smi

Thu Feb 26 10:59:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P0             27W /   70W |   11339MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [47]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision 
import torchvision.transforms as transforms

from torchvision.models import mobilenet_v2

In [48]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [49]:
# transforms
transform_train = transforms.Compose(
    [
        transforms.Resize(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ]
)

transform_test = transforms.Compose(
    [
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ]
)

In [50]:
# dataset
trainset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform_train
)

testset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform_test
)

trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=64, shuffle=True
)

testloader = torch.utils.data.DataLoader(
    testset, batch_size=64, shuffle=False
)

In [51]:
model = mobilenet_v2(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [52]:
# Replace classifier for 10 classes
model.classifier[1] = nn.Linear(model.last_channel, 10)
model = model.to(device)

In [53]:
# loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [54]:
# training
for epoch in range(5):
    model.train()
    running_loss = 0.0

    for images, labels in trainloader:
        images,labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        
    print(f"Epoch {epoch + 1}, Loss: {running_loss / len(trainloader)}")

print("Training finished")

Epoch 1, Loss: 0.5231210450496515
Epoch 2, Loss: 0.35124753324119634
Epoch 3, Loss: 0.2891064774235496
Epoch 4, Loss: 0.2556046117859347
Epoch 5, Loss: 0.23174996085731728
Training finished


In [55]:
# testing
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total}%")

Test Accuracy: 89.84%


In [ ]:
# parameter count
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Total params:", total_params)
print("Trainable params:", trainable_params)

Total params: 2236682
Trainable params: 2236682


In [58]:
import time

model.eval()
dummy = torch.randn(1, 3, 224, 224).to(device)

# warm-up
for _ in range(10):
    _ = model(dummy)

torch.cuda.synchronize()

start = time.time()
for _ in range(100):
    _ = model(dummy)
torch.cuda.synchronize()
end = time.time()

latency = (end - start) / 100
print(f"Latency: {latency * 1000:.2f} ms")

Latency: 6.99 ms


In [59]:
from fvcore.nn import FlopCountAnalysis

model.eval()
dummy = torch.randn(1, 3, 224, 224).to(device)

flops = FlopCountAnalysis(model, dummy)
print(flops.total())

312926016
